In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_csv('solver_runs.csv')

In [3]:
df.head()

,grid_size,iterations,time_taken,l2_error
0,50,85,0.322047,NaN
1,100,162,0.503018,NaN
2,150,240,1.019925,NaN
3,200,319,2.034440,NaN
4,250,397,2.835138,NaN


In [4]:
df.drop('l2_error',axis=1,inplace=True)

In [5]:
df.head(2)

,grid_size,iterations,time_taken
0,50,85,0.322047
1,100,162,0.503018


In [6]:
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures

X_i = df[['grid_size']].values
Y1 = df[['iterations']].values
Y2 = df[['time_taken']].values

# Polynomial transformation
poly = PolynomialFeatures(degree=3)
X_t = poly.fit_transform(X_i)

# Linear model for iterations
iter_model = LinearRegression()
iter_model.fit(X_i, Y1)

print(iter_model.intercept_)

# Polynomial model for time_taken
time_model = LinearRegression()
time_model.fit(X_t, Y2)

[1.4538874]


,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


In [8]:
pip install joblib

Note: you may need to restart the kernel to use updated packages.


You should consider upgrading via the 'c:\Users\sjija\AppData\Local\Programs\Python\Python310\python.exe -m pip install --upgrade pip' command.


In [9]:
import joblib

joblib.dump(iter_model, "iteration_model.pkl")
joblib.dump(time_model, "time_model.pkl")
joblib.dump(poly, "poly_transform.pkl")   # important for polynomial model

['poly_transform.pkl']

In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
import joblib
import os

def train_and_save_models():
    file_path = 'solver_runs.csv'
    if not os.path.exists(file_path):
        print("Error: solver_runs.csv not found. Run some simulations first!")
        return

    # 1. Load Data
    df = pd.read_csv(file_path)
    
    # 2. Preprocessing: Remove Statistical Outliers (IQR Method)
    # This prevents one-off lag spikes from ruining your n^3 time curve
    Q1 = df['time_taken'].quantile(0.25)
    Q3 = df['time_taken'].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    df_cleaned = df[(df['time_taken'] >= lower_bound) & (df['time_taken'] <= upper_bound)]
    df_cleaned = df_cleaned.sort_values("grid_size").drop_duplicates("grid_size")

    if len(df_cleaned) < 3:
        print("Not enough unique data points to train a reliable model.")
        return

    X = df_cleaned[['grid_size']].values
    y_iters = df_cleaned['iterations'].values
    y_time = df_cleaned['time_taken'].values

    # 3. Train Iteration Model: Linear O(n)
    iter_model = LinearRegression()
    iter_model.fit(X, y_iters)
    
    # 4. Train Time Model: Cubic O(n^3)
    poly_features = PolynomialFeatures(degree=3)
    X_poly = poly_features.fit_transform(X)
    time_model = LinearRegression()
    time_model.fit(X_poly, y_time)

    # 5. Save Models and the Poly Transformer
    joblib.dump(iter_model, 'iter_model.pkl')
    joblib.dump(time_model, 'time_model.pkl')
    joblib.dump(poly_features, 'poly_transformer.pkl')
    
    print(f"Models trained successfully using {len(df_cleaned)} data points.")
    print(f"Iteration R^2: {iter_model.score(X, y_iters):.4f}")
    print(f"Time R^2: {time_model.score(X_poly, y_time):.4f}")

if __name__ == "__main__":
    train_and_save_models()

Models trained successfully using 23 data points.
Iteration R^2: 0.9996
Time R^2: 0.9807
